In [2]:
# -*- coding: utf-8 -*-
"""
ecg_full_toolkit_FINAL.py
================================================================================
ЕДИНЫЙ файл, объединяющий весь код, использованный при разработке и проверке
итогового пайплайна детекции R/P/T-зубцов ЭКГ.

Структура файла (4 раздела):

  РАЗДЕЛ A — ФИНАЛЬНЫЙ АЛГОРИТМ (universal_ecg_pt_FINAL.py)
             Основной, рекомендуемый к использованию код. История версий
             (v1 - ноутбук с фиксированным порогом, v2 - адаптивный порог,
             v3 - добавлен выбор отведения по QRS-энергии) описана в
             комментарии перед разделом ниже.

  РАЗДЕЛ B — НЕЗАВИСИМЫЙ ДЕТЕКТОР ДЛЯ ПЕРЕКРЁСТНОЙ ПРОВЕРКИ (ecg_pipeline.py)
             Второй, независимо реализованный (по другим принципам —
             Pan-Tompkins-подобная схема) детектор R/P/T. Используется не
             как основной инструмент анализа, а как "внешний свидетель":
             если оба независимых алгоритма совпадают — это содержательное
             свидетельство корректности на записях без готового эталона.
             Функции — с приставкой crossval_.

  РАЗДЕЛ C — СИНТЕТИЧЕСКИЙ ЭТАЛОН (synthetic_ecg.py)
             Генератор ЭКГ-подобного сигнала с ТОЧНО известными (заданными
             как параметр генерации, а не измеренными) положениями R-зубцов.
             Единственный способ получить численные Se/+P/F1 без доступа к
             аннотированным клиническим базам (physionet.org и т.п.
             недоступны из этой среды выполнения).

  РАЗДЕЛ D — РАСЧЁТ МЕТРИК И ВАЛИДАЦИЯ (validate_r_peak_accuracy.py)
             Сопоставление найденных пиков с эталоном/друг с другом,
             расчёт Se/+P/F1, запуск полной проверки (синтетика + реальные
             записи) одной командой.

Пример использования — см. блок if __name__ == "__main__" в конце файла.
================================================================================
"""
!pip install pyEDFlib PyWavelets -q
import glob
import json
import math
import os
import warnings

import numpy as np
import pyedflib
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks, cheby2, filtfilt, welch
from scipy import signal as _sig


# ══════════════════════════════════════════════════════════════════════════
# РАЗДЕЛ A. ФИНАЛЬНЫЙ АЛГОРИТМ  (universal_ecg_pt_FINAL.py)
# ══════════════════════════════════════════════════════════════════════════
#
# В данной версии добавлен select_best_lead(): выбор отведения по
# максимуму QRS-энергии вместо первого канала с 'ecg' в названии.
# Устраняет ложный диагноз тахиаритмии на записи 0027_ecg.edf (канал 'ECG'
# там даёт R=753 из-за T-волны, ошибочно принятой
# за R; каналы 'ECG2'/'ECG3' — правильные 491,
# ЧСС=48.6, брадикардия, а не аритмия).

_UNIT_TO_MV = {
    "v": 1000.0, "volt": 1000.0, "volts": 1000.0,
    "mv": 1.0, "millivolt": 1.0, "millivolts": 1.0,
    "uv": 0.001, "µv": 0.001, "microvolt": 0.001, "microvolts": 0.001,
}


def find_ecg_channel(labels, preferred=None):
    """Находит индекс ЭКГ-канала по названию. Если preferred задан и
    найден — берёт его; иначе первый канал, содержащий 'ECG'/'ЭКГ'."""
    if preferred is not None:
        for i, lab in enumerate(labels):
            if preferred.lower() in lab.lower():
                return i
    for i, lab in enumerate(labels):
        if "ecg" in lab.lower() or "экг" in lab.lower():
            return i
    return 0


def read_edf_channel_mV(path: str, channel_name: str = None):
    """Универсальное чтение одного канала EDF в мВ: находит ЭКГ-канал по
    названию, приводит единицы измерения к мВ по метаданным заголовка."""
    f = pyedflib.EdfReader(path)
    try:
        labels = f.getSignalLabels()
        ch = find_ecg_channel(labels, channel_name)
        fs = float(f.getSampleFrequency(ch))
        sig = f.readSignal(ch).astype(np.float64)
        unit = f.getPhysicalDimension(ch).strip().lower()
        scale = _UNIT_TO_MV.get(unit)
        if scale is None:
            warnings.warn(f"Неизвестная единица измерения '{unit}' — "
                           f"сигнал возвращён без масштабирования.")
            scale = 1.0
        return sig * scale, fs, labels[ch]
    finally:
        f.close()


def select_best_lead(path: str) -> str:
    """[v3] Автоматический выбор ЭКГ-отведения по максимуму доли
    спектральной энергии сигнала в полосе QRS (5-15 Гц) — вместо первого
    канала, где просто есть подстрока 'ecg'/'экг' в названии. См. пояснение
    и обоснование в шапке РАЗДЕЛА A выше и в отчёте по записи 0027_ecg.edf."""
    f = pyedflib.EdfReader(path)
    try:
        labels = f.getSignalLabels()
        fs = float(f.getSampleFrequency(0))
        raw = [f.readSignal(i).astype(np.float64) for i in range(f.signals_in_file)]
    finally:
        f.close()

    def qrs_energy_ratio(x):
        freqs, pxx = welch(x, fs=fs, nperseg=int(4 * fs))
        band = (freqs >= 5) & (freqs <= 15)
        return np.sum(pxx[band]) / (np.sum(pxx) + 1e-12)

    scores = [qrs_energy_ratio(x) for x in raw]
    return labels[int(np.argmax(scores))]


def _sine_wavelet(length: int) -> np.ndarray:
    t = np.linspace(0, math.pi, length)
    w = np.sin(t)
    return w - w.mean()


def filter_ecg(sig_mV: np.ndarray, fs: float, fc: float = 0.5, order: int = 5):
    """Фильтр Чебышёва II рода (ФВЧ, fc=0.5 Гц) — устраняет дрейф изолинии
    перед детекцией R, не искажая форму QRS-комплекса."""
    nyq = fs / 2.0
    b, a = cheby2(order, rs=40, Wn=fc / nyq, btype="high", analog=False)
    return filtfilt(b, a, sig_mV)


def _wavelet_image(sig: np.ndarray, w: np.ndarray) -> np.ndarray:
    L, N, half = len(w), len(sig), len(w) // 2
    out = np.zeros(N)
    for i in range(half, N - half):
        out[i] = float(np.dot(sig[i - half:i - half + L], w))
    return out


def detect_r_peaks(ecg_mV: np.ndarray, fs: float, min_rr_ms: float = 300,
                    k_threshold: float = 2.0) -> np.ndarray:
    """[v2] Детекция R-зубцов с АДАПТИВНЫМ порогом (mean+k*std от
    вейвлет-образа сигнала), пересчитываемым заново для каждой записи —
    работает при любой амплитуде сигнала (в отличие от v1 с фиксированным
    порогом 460 мкВ, откалиброванным под одну конкретную запись)."""
    wlen = max(3, int(0.12 * fs))
    wav = _wavelet_image(ecg_mV, _sine_wavelet(wlen))
    level = wav.mean() + k_threshold * wav.std()
    above = np.where(wav > level)[0]
    if len(above) == 0:
        return np.array([], dtype=int)

    min_gap = int(min_rr_ms / 1000.0 * fs)
    peaks, group = [], [above[0]]
    for idx in above[1:]:
        if idx - group[-1] <= 1:
            group.append(idx)
        else:
            peaks.append(group[0] + int(np.argmax(wav[group])))
            group = [idx]
    peaks.append(group[0] + int(np.argmax(wav[group])))

    out = [peaks[0]]
    for pk in peaks[1:]:
        if pk - out[-1] >= min_gap:
            out.append(pk)
    return np.array(out, dtype=int)


def check_double_counting(r_idx: np.ndarray, fs: float, tol: float = 0.15) -> bool:
    """Самопроверка на задвоение: True, если RR-интервалы регулярно
    чередуются короткий/длинный (признак того, что порог всё ещё ловит
    два срабатывания на один удар)."""
    if len(r_idx) < 5:
        return False
    rr = np.diff(r_idx) / fs
    short, long_ = rr[0::2], rr[1::2]
    n = min(len(short), len(long_))
    if n < 2:
        return False
    is_alt = (np.std(short[:n]) / np.mean(short[:n]) < tol and
              np.std(long_[:n]) / np.mean(long_[:n]) < tol and
              np.mean(short[:n]) < 0.6 * np.mean(long_[:n]))
    return bool(is_alt)


def filter_qrs(ecg_mV: np.ndarray, r_idx: np.ndarray, fs: float) -> np.ndarray:
    """Вырезает QRS-комплекс линейной интерполяцией в окне [-50мс,+100мс] —
    чтобы крупный QRS не перекрывал существенно более мелкие P и T."""
    half, qrs_e = int(0.05 * fs), int(0.10 * fs)
    out = ecg_mV.copy()
    for r in r_idx:
        i0, i1 = max(0, r - half), min(len(ecg_mV) - 1, r + qrs_e)
        if i1 - i0 < 2:
            continue
        x = np.arange(i0, i1 + 1)
        out[i0:i1 + 1] = np.interp(x, [i0, i1], [ecg_mV[i0], ecg_mV[i1]])
    return out


def _halfwave_wavelet(lam: int = 17) -> np.ndarray:
    if lam % 2 == 0:
        lam += 1
    t = np.linspace(0, math.pi, lam)
    pos = np.sin(t)
    area = float(np.trapezoid(pos)) if hasattr(np, "trapezoid") else float(np.trapz(pos))
    wing = max(1, lam // 4)
    w = np.zeros(lam + 2 * wing)
    w[:wing] = -area / (2 * wing)
    w[wing:wing + lam] = pos
    w[wing + lam:] = -area / (2 * wing)
    return w - w.mean()


def detect_pt_peaks(ecg_f: np.ndarray, r_idx: np.ndarray, fs: float, lam: int = None,
                     k_local: int = 15, k_mult: float = 0.3):
    """P/T-зубцы: окна поиска — доли ФАКТИЧЕСКОГО соседнего RR (не
    фиксированные мс), порог — скользящая МЕДИАНА отклика по ±k_local
    соседним ударам. Ширина вейвлета (lam) масштабируется по fs (~85 мс).

    Пороги для P и T считаются РАЗДЕЛЬНО (медиана P-откликов
    и медиана T-откликов по отдельности), а не от общей статистики
    max(P,T). Проблема общей статистики обнаружена на записи 0028_ecg.edf:
    начиная примерно с 310-й секунды записи амплитуда T-волны падает почти
    в 3.5 раза, а амплитуда P-волны не меняется совсем. Общий порог
    (медиана max(P,T)) в этом случае продолжает ориентироваться на
    (по-прежнему крупную) P и остаётся завышенным для T — T-зубец
    пропускался в ~24% ударов на этом участке. Раздельные пороги
    устраняют эту зависимость P и T друг от друга."""
    if lam is None:
        lam = max(5, int(0.085 * fs))
    wav = _wavelet_image(ecg_f, _halfwave_wavelet(lam))

    n = len(r_idx)
    p_windows, t_windows = [], []
    pv_arr, tv_arr = np.zeros(n), np.zeros(n)
    for k in range(n):
        r = r_idx[k]
        r_prev = r_idx[k - 1] if k > 0 else max(0, r - int(fs))
        r_next = r_idx[k + 1] if k < n - 1 else min(len(ecg_f) - 1, r + int(fs))

        rr = r - r_prev
        ps, pe = r_prev + max(1, rr // 6), r - max(1, rr // 6)
        pv = wav[ps:pe].max() if ps < pe else 0.0

        rnn = r_next - r
        ts, te = r + max(1, rnn // 5), r + min(rnn - 1, int(0.75 * rnn))
        tv = wav[ts:te].max() if ts < te <= len(wav) else 0.0

        p_windows.append((ps, pe, pv))
        t_windows.append((ts, te, tv))
        pv_arr[k] = pv
        tv_arr[k] = tv

    def rolling_median(vals):
        out = np.zeros(n)
        for k in range(n):
            lo, hi = max(0, k - k_local), min(n, k + k_local + 1)
            out[k] = np.median(vals[lo:hi])
        return out

    p_scale = rolling_median(pv_arr)
    t_scale = rolling_median(tv_arr)

    p_idx, t_idx = [], []
    for k in range(n):
        thr_p = k_mult * p_scale[k]
        thr_t = k_mult * t_scale[k]
        ps, pe, pv = p_windows[k]
        if ps < pe and pv > thr_p:
            p_idx.append(ps + int(np.argmax(wav[ps:pe])))
        ts, te, tv = t_windows[k]
        if ts < te <= len(wav) and tv > thr_t:
            t_idx.append(ts + int(np.argmax(wav[ts:te])))
    return np.array(p_idx, dtype=int), np.array(t_idx, dtype=int)


def analyze_edf(path: str, channel_name: str = None, window_s: tuple = None,
                 verbose: bool = True, auto_select_lead: bool = True):
    """Полный анализ одного EDF-файла: читает канал -> детектирует R ->
    вырезает QRS -> детектирует P/T -> проверяет на задвоение.

    auto_select_lead=True (по умолчанию): если channel_name не задан явно,
    отведение выбирается автоматически по QRS-энергии (select_best_lead).
    Чтобы воспроизвести поведение v2 (первый канал с 'ecg' в названии) —
    вызовите с auto_select_lead=False."""
    if channel_name is None and auto_select_lead:
        channel_name = select_best_lead(path)

    sig_mV, fs, used_label = read_edf_channel_mV(path, channel_name)

    if window_s is not None:
        i0, i1 = int(window_s[0] * fs), int(window_s[1] * fs)
        sig_mV = sig_mV[i0:i1]
    else:
        i0 = 0

    ecg_hf = filter_ecg(sig_mV, fs)
    r_idx = detect_r_peaks(ecg_hf, fs)
    doubled = check_double_counting(r_idx, fs)
    if doubled and verbose:
        warnings.warn(f"[{path}] Похоже на задвоение R-пиков - проверьте вручную.")

    ecg_f = filter_qrs(ecg_hf, r_idx, fs)
    p_idx, t_idx = detect_pt_peaks(ecg_f, r_idx, fs)

    if verbose:
        dur_s = len(sig_mV) / fs
        hr = 60.0 / np.mean(np.diff(r_idx) / fs) if len(r_idx) > 1 else float("nan")
        print(f"[{path}] канал={used_label} fs={fs:.0f}Гц окно={dur_s:.1f}с | "
              f"R={len(r_idx)} P={len(p_idx)} T={len(t_idx)} | "
              f"ЧСС≈{hr:.1f} уд/мин | задвоение={'ДА !!!' if doubled else 'нет'}")

    return {
        "signal_mV": sig_mV, "fs": fs, "channel": used_label,
        "r_idx": r_idx, "p_idx": p_idx, "t_idx": t_idx,
        "ecg_qrs_removed": ecg_f, "offset_samples": i0,
        "double_counting_suspected": doubled,
    }


def plot_analysis(path: str, res_full: dict, res_win: dict, window_s: tuple,
                   save_path: str = None):
    """Строит рисунок по тем же данным, что и analyze_edf (числа в
    заголовке и в print() гарантированно совпадают)."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fs = res_full["fs"]
    r_idx_full = res_full["r_idx"]
    rr_s = np.diff(r_idx_full) / fs
    t_mid = (r_idx_full[:-1] + r_idx_full[1:]) / 2 / fs / 60
    hr = 60.0 / rr_s

    sig_raw = res_win["signal_mV"]
    sig_qrs_removed = res_win["ecg_qrs_removed"]
    fs_w = res_win["fs"]
    r_w, p_w, t_w = res_win["r_idx"], res_win["p_idx"], res_win["t_idx"]
    t_axis = np.arange(len(sig_raw)) / fs_w

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(3, 1, height_ratios=[1.3, 1, 1])

    ax1 = fig.add_subplot(gs[0])
    ax1.plot(t_axis, sig_raw, lw=0.8, color="tab:blue", label="ЭКГ (после ФВЧ)")
    ax1.scatter(r_w / fs_w, sig_raw[r_w], color="red", s=60, zorder=5, label=f"R (n={len(r_w)})")
    if len(p_w):
        ax1.scatter(p_w / fs_w, sig_qrs_removed[p_w], color="green", marker="^", s=55, zorder=5, label=f"P (n={len(p_w)})")
    if len(t_w):
        ax1.scatter(t_w / fs_w, sig_qrs_removed[t_w], color="blue", marker="v", s=55, zorder=5, label=f"T (n={len(t_w)})")
    ax1.set_title(f"{path} — канал {res_win['channel']}, фрагмент {window_s[0]}-{window_s[1]} с "
                  f"| R={len(r_w)} P={len(p_w)} T={len(t_w)}")
    ax1.set_xlabel("Время, с"); ax1.set_ylabel("мВ")
    ax1.legend(loc="upper right", fontsize=8); ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[1])
    ax2.plot(rr_s, lw=0.7, color="tab:orange")
    ax2.scatter(np.arange(len(rr_s)), rr_s, s=4, color="tab:orange")
    ax2.axhline(np.median(rr_s), color="gray", ls="--", lw=1, label=f"медиана={np.median(rr_s):.2f}с")
    ax2.set_title(f"RR-интервалы по всей записи (n={len(rr_s)}) — "
                  f"{'ЗАДВОЕНИЕ ЗАПОДОЗРЕНО' if res_full['double_counting_suspected'] else 'чередования короткий/длинный нет'}")
    ax2.set_xlabel("Номер удара"); ax2.set_ylabel("RR, с")
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[2])
    ax3.plot(t_mid, hr, lw=0.5, color="tab:purple")
    ax3.set_title(f"Динамика ЧСС по всей записи (R={len(r_idx_full)}, "
                  f"P={len(res_full['p_idx'])}, T={len(res_full['t_idx'])}, "
                  f"медиана ЧСС={np.median(hr):.1f} уд/мин)")
    ax3.set_xlabel("Время, мин"); ax3.set_ylabel("ЧСС, уд/мин")
    ax3.grid(alpha=0.3)

    fig.suptitle(f"{path} — детекция R/P/T (финальный пайплайн)", fontsize=13, y=1.0)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=140, bbox_inches="tight")
        plt.close(fig)
        print(f"  -> сохранено: {save_path}")
    return fig


def run_full_report(path: str, window_s: tuple = (0, 10), save_dir: str = "."):
    """Считает полную запись (для RR/ЧСС и проверки на задвоение) и одно
    окно window_s (для наглядной картинки) — печатает и строит рисунок
    строго по этим двум результатам."""
    print(f"\n=== {path} ===")
    res_full = analyze_edf(path, verbose=True)
    res_win = analyze_edf(path, window_s=window_s, verbose=True)
    save_path = f"{save_dir}/final_{os.path.basename(path).replace('.edf', '')}.png"
    plot_analysis(path, res_full, res_win, window_s, save_path=save_path)
    return res_full, res_win


# ══════════════════════════════════════════════════════════════════════════
# РАЗДЕЛ B. НЕЗАВИСИМЫЙ ДЕТЕКТОР ДЛЯ ПЕРЕКРЁСТНОЙ ПРОВЕРКИ (ecg_pipeline.py)
# ══════════════════════════════════════════════════════════════════════════
#
# Построен на ДРУГИХ принципах (Pan-Tompkins: производная -> квадрат ->
# скользящее интегрирование), чем РАЗДЕЛ A (согласованная фильтрация
# вейвлетом). Используется исключительно для перекрёстной проверки: если
# два независимо реализованных алгоритма совпадают на реальной записи без
# готового эталона — это содержательное (хотя не абсолютное) свидетельство
# корректности обоих.

def crossval_bandpass_filter(x, fs, low=0.5, high=40.0, order=4):
    nyq = fs / 2
    b, a = _sig.butter(order, [low / nyq, high / nyq], btype='band')
    return _sig.filtfilt(b, a, x)


def crossval_notch_filter(x, fs, freq=50.0, q=30.0):
    b, a = _sig.iirnotch(freq / (fs / 2), q)
    return _sig.filtfilt(b, a, x)


def crossval_detect_r_peaks(x, fs):
    """Pan-Tompkins-подобная детекция: производная -> квадрат -> скользящее
    интегрирование (окно 150мс) -> адаптивный порог (mean+0.5*std) ->
    уточнение положения на исходном сигнале -> рефрактерный период 300мс."""
    diff = np.diff(x, prepend=x[0])
    squared = diff ** 2
    win = int(0.150 * fs)
    integrated = np.convolve(squared, np.ones(win) / win, mode='same')

    min_rr_samples = int(0.30 * fs)
    peaks, _ = find_peaks(integrated, distance=min_rr_samples,
                           height=np.mean(integrated) + 0.5 * np.std(integrated))

    refine_w = int(0.04 * fs)
    refined = []
    for pk in peaks:
        lo, hi = max(0, pk - refine_w), min(len(x), pk + refine_w)
        local = x[lo:hi]
        refined.append(lo + int(np.argmax(np.abs(local))))
    refined = np.array(sorted(set(refined)))

    if len(refined) > 1:
        keep = [refined[0]]
        for pk in refined[1:]:
            if pk - keep[-1] >= min_rr_samples:
                keep.append(pk)
        refined = np.array(keep)
    return refined


def crossval_hrv_time_domain(rpeaks, fs):
    """Временные показатели ВСР: mean RR, SDNN, RMSSD, pNN50, ЧСС."""
    if len(rpeaks) < 3:
        return None
    rr = np.diff(rpeaks) / fs * 1000.0
    valid = (rr > 300) & (rr < 2000)
    rr = rr[valid]
    if len(rr) < 2:
        return None
    diff_rr = np.diff(rr)
    mean_rr = np.mean(rr)
    return dict(
        mean_rr_ms=mean_rr, sdnn_ms=np.std(rr, ddof=1),
        rmssd_ms=np.sqrt(np.mean(diff_rr ** 2)),
        pnn50_pct=100.0 * np.sum(np.abs(diff_rr) > 50) / len(diff_rr) if len(diff_rr) else np.nan,
        mean_hr_bpm=60000.0 / mean_rr, n_beats=len(rpeaks), n_valid_rr=len(rr),
    )


def crossval_find_p_and_t_waves(x, rpeaks, fs):
    """P-зубец в окне PR 80-200мс до R; T-зубец в окне 100-400мс после R
    (независимая, более простая от РАЗДЕЛА A логика окон — намеренно
    иная, чтобы служить настоящей перекрёстной проверкой, а не копией)."""
    p_amps, t_amps = [], []
    for r in rpeaks:
        p_lo, p_hi = r - int(0.200 * fs), r - int(0.080 * fs)
        if p_lo >= 0:
            seg = x[p_lo:p_hi]
            if len(seg) > 0:
                p_amps.append(seg[np.argmax(seg)])
        t_lo, t_hi = r + int(0.100 * fs), r + int(0.400 * fs)
        if t_hi <= len(x):
            seg = x[t_lo:t_hi]
            if len(seg) > 0:
                t_amps.append(seg[np.argmax(np.abs(seg))])
    out = {"p_detect_rate_pct": 100.0 * len(p_amps) / len(rpeaks) if len(rpeaks) else 0.0}
    if t_amps:
        out["t_detect_rate_pct"] = 100.0 * len(t_amps) / len(rpeaks)
    return out


def crossval_qrs_energy_ratio(xf, fs):
    """Доля спектральной энергии в полосе QRS (5-15Гц) — используется и
    для выбора отведения (см. select_best_lead в РАЗДЕЛЕ A), и как общая
    метрика качества сигнала для этого детектора."""
    freqs, pxx = _sig.welch(xf, fs=fs, nperseg=int(4 * fs))
    band = (freqs >= 5) & (freqs <= 15)
    return np.sum(pxx[band]) / (np.sum(pxx) + 1e-12)


# ══════════════════════════════════════════════════════════════════════════
# РАЗДЕЛ C. СИНТЕТИЧЕСКИЙ ЭТАЛОН (synthetic_ecg.py)
# ══════════════════════════════════════════════════════════════════════════
#
# Момент R любого удара задаётся АНАЛИТИЧЕСКИ (параметр генерации, а не
# результат измерения/детекции) — истинный, безошибочный эталон. Нужен,
# т.к. аннотированные клинические базы (physionet.org) недоступны
# (не входит в список разрешённых сетевых доменов).

def _gauss(t, center, width, amp):
    return amp * np.exp(-0.5 * ((t - center) / width) ** 2)


def make_beat_template(fs, amp_scale=1.0):
    """Форма одного удара (P,Q,R,S,T — сумма гауссовых "холмов") относительно
    момента R (t=0). Амплитуды подобраны по порядку величины физиологической
    ЭКГ: P~0.15мВ, R~1.2мВ, T~0.3мВ."""
    def one_beat(t_ms):
        s = np.zeros_like(t_ms)
        s += _gauss(t_ms, -160, 25, 0.15 * amp_scale)   # P
        s += _gauss(t_ms, -40, 8, -0.10 * amp_scale)    # Q
        s += _gauss(t_ms, 0, 10, 1.20 * amp_scale)      # R
        s += _gauss(t_ms, 35, 9, -0.25 * amp_scale)     # S
        s += _gauss(t_ms, 250, 45, 0.30 * amp_scale)    # T
        return s
    return one_beat


def generate_synthetic_ecg(fs=500.0, duration_s=120.0, hr_bpm=70.0,
                            hrv_pct=3.0, noise_level="clean", seed=0):
    """Синтетическая одноканальная ЭКГ с истинной (заданной) последова-
    тельностью R. noise_level: 'clean' | 'realistic' | 'noisy'.
    Возвращает (signal_mV, fs, true_r_idx)."""
    rng = np.random.default_rng(seed)
    mean_rr_s = 60.0 / hr_bpm
    n_samples = int(duration_s * fs)

    r_times = [0.3]
    while r_times[-1] < duration_s - 0.5:
        rr = mean_rr_s * (1 + rng.normal(0, hrv_pct / 100.0))
        rr = max(0.3, rr)
        r_times.append(r_times[-1] + rr)
    r_times = np.array(r_times[:-1])
    true_r_idx = np.round(r_times * fs).astype(int)
    true_r_idx = true_r_idx[true_r_idx < n_samples]

    one_beat = make_beat_template(fs)
    t_axis_ms = (np.arange(n_samples) / fs) * 1000.0
    signal = np.zeros(n_samples)

    half_lo_ms, half_hi_ms = 300, 450
    for r in true_r_idx:
        r_ms = r / fs * 1000.0
        lo = max(0, int(r - half_lo_ms / 1000.0 * fs))
        hi = min(n_samples, int(r + half_hi_ms / 1000.0 * fs))
        signal[lo:hi] += one_beat(t_axis_ms[lo:hi] - r_ms)

    if noise_level == "clean":
        pass
    elif noise_level == "realistic":
        signal += rng.normal(0, 0.02, n_samples)
        signal += 0.10 * np.sin(2 * np.pi * 0.05 * np.arange(n_samples) / fs + rng.uniform(0, 2 * np.pi))
    elif noise_level == "noisy":
        signal += rng.normal(0, 0.06, n_samples)
        signal += 0.25 * np.sin(2 * np.pi * 0.08 * np.arange(n_samples) / fs + rng.uniform(0, 2 * np.pi))
        n_art = int(duration_s / 20)
        for _ in range(n_art):
            pos = rng.integers(0, n_samples - 100)
            signal[pos:pos + 100] += rng.normal(0, 0.4, 100)
    else:
        raise ValueError(noise_level)

    return signal, fs, true_r_idx


# ══════════════════════════════════════════════════════════════════════════
# РАЗДЕЛ D. РАСЧЁТ МЕТРИК И ВАЛИДАЦИЯ (validate_r_peak_accuracy.py)
# ══════════════════════════════════════════════════════════════════════════

SYNTHETIC_SCENARIOS = [
    dict(name="Чистый сигнал, ЧСС=60", hr_bpm=60, noise_level="clean", hrv_pct=1, seed=1),
    dict(name="Чистый сигнал, ЧСС=100", hr_bpm=100, noise_level="clean", hrv_pct=1, seed=2),
    dict(name="Реалистичный шум, ЧСС=70 (норма)", hr_bpm=70, noise_level="realistic", hrv_pct=5, seed=3),
    dict(name="Реалистичный шум, ЧСС=45 (брадикардия)", hr_bpm=45, noise_level="realistic", hrv_pct=5, seed=4),
    dict(name="Реалистичный шум, ЧСС=130 (тахикардия)", hr_bpm=130, noise_level="realistic", hrv_pct=8, seed=5),
    dict(name="Сильный шум+артефакты, ЧСС=75", hr_bpm=75, noise_level="noisy", hrv_pct=6, seed=6),
    dict(name="Сильный шум+артефакты, ЧСС=160 (тахикардия)", hr_bpm=160, noise_level="noisy", hrv_pct=6, seed=7),
]


def match_peaks(true_idx, det_idx, fs, tol_ms):
    """Жадное сопоставление детектированных пиков с эталонными (каждый
    эталонный пик используется не более одного раза). Возвращает
    TP, FN, FP, список ошибок (мс) для найденных совпадений."""
    tol = tol_ms / 1000.0 * fs
    true_idx, det_idx = np.sort(true_idx), np.sort(det_idx)
    used_true = np.zeros(len(true_idx), dtype=bool)
    used_det = np.zeros(len(det_idx), dtype=bool)
    errors_ms = []
    j = 0
    for i, t in enumerate(true_idx):
        best_j, best_d = -1, None
        k = j
        while k < len(det_idx) and det_idx[k] < t - tol:
            k += 1
        m = k
        while m < len(det_idx) and det_idx[m] <= t + tol:
            if not used_det[m]:
                d = abs(det_idx[m] - t)
                if best_d is None or d < best_d:
                    best_d, best_j = d, m
            m += 1
        if best_j >= 0:
            used_det[best_j] = True
            used_true[i] = True
            errors_ms.append((det_idx[best_j] - t) / fs * 1000.0)
        j = k
    TP = int(used_true.sum())
    FN = int((~used_true).sum())
    FP = int((~used_det).sum())
    return TP, FN, FP, errors_ms


def se_pp_f1(TP, FN, FP):
    se = TP / (TP + FN) if (TP + FN) > 0 else float('nan')
    pp = TP / (TP + FP) if (TP + FP) > 0 else float('nan')
    f1 = 2 * se * pp / (se + pp) if (se + pp) > 0 else float('nan')
    return se, pp, f1


def run_synthetic_validation(save_json="results/synthetic_validation.json"):
    """Проверяет detect_r_peaks (РАЗДЕЛ A, финальный алгоритм) на всех
    сценариях синтетического эталона (РАЗДЕЛ D). Возвращает список
    результатов по каждому сценарию, при желании сохраняет в JSON."""
    results = []
    for sc in SYNTHETIC_SCENARIOS:
        sig, fs, true_r = generate_synthetic_ecg(
            fs=500.0, duration_s=180.0, hr_bpm=sc["hr_bpm"],
            hrv_pct=sc["hrv_pct"], noise_level=sc["noise_level"], seed=sc["seed"])
        ecg_hf = filter_ecg(sig, fs)
        det_r = detect_r_peaks(ecg_hf, fs)
        row = dict(scenario=sc["name"], n_true=len(true_r), n_detected=len(det_r))
        for tol in (50, 150):
            TP, FN, FP, errs = match_peaks(true_r, det_r, fs, tol_ms=tol)
            se, pp, f1 = se_pp_f1(TP, FN, FP)
            row[f"tol{tol}ms"] = dict(TP=TP, FN=FN, FP=FP, Se=se, PP=pp, F1=f1,
                                       mean_abs_err_ms=float(np.mean(np.abs(errs))) if errs else None)
        results.append(row)
        print(f"{sc['name']:45s} | true={len(true_r):4d} det={len(det_r):4d} | "
              f"Se={row['tol50ms']['Se']:.4f} PP={row['tol50ms']['PP']:.4f} F1={row['tol50ms']['F1']:.4f}")
    if save_json:
        os.makedirs(os.path.dirname(save_json), exist_ok=True)
        with open(save_json, "w") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
    return results


def run_cross_validation(data_dir, save_json="results/cross_validation.json"):
    """Перекрёстная проверка на всех EDF-файлах в data_dir: финальный
    алгоритм (РАЗДЕЛ A) vs независимый детектор (РАЗДЕЛ B). Отведение
    выбирается автоматически (select_best_lead) для обоих детекторов."""
    files = sorted(glob.glob(os.path.join(data_dir, "*.edf")))
    results = []
    for fp in files:
        lead = select_best_lead(fp)
        sig_mV, fs, _ = read_edf_channel_mV(fp, lead)

        ecg_hf = filter_ecg(sig_mV, fs)
        r_final = detect_r_peaks(ecg_hf, fs)
        doubled = check_double_counting(r_final, fs)

        xf_cross = crossval_notch_filter(crossval_bandpass_filter(sig_mV, fs=fs), fs=fs)
        r_cross = crossval_detect_r_peaks(xf_cross, fs=fs)

        TP, FN, FP, errs = match_peaks(r_cross, r_final, fs, tol_ms=50)
        se, pp, f1 = se_pp_f1(TP, FN, FP)

        row = dict(file=os.path.basename(fp), lead=lead,
                   n_final=len(r_final), n_crossval=len(r_cross),
                   doubled=doubled, agreement_pct=round(se * 100, 2))
        results.append(row)
        print(f"{row['file']:15s} лидер={lead:6s} R(финал)={len(r_final):4d} "
              f"R(перекр.)={len(r_cross):4d} согласие={row['agreement_pct']:.2f}% "
              f"задвоение={'ДА' if doubled else 'нет'}")

    if save_json:
        os.makedirs(os.path.dirname(save_json), exist_ok=True)
        with open(save_json, "w") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
    return results


def run_batch_analysis(data_dir, save_json="results/batch_rpt_analysis.json",
                        make_illustrations=True, fig_dir="figures", window_s=(0, 10)):
    """
    Полный анализ R/P/T КАЖДОГО EDF-файла в data_dir (Раздел A, финальный
    алгоритм) — с печатью результата по каждому файлу отдельно. Это то,
    чего не хватало в предыдущей версии: run_cross_validation печатает
    только сравнение R с независимым детектором, а не R/P/T по каждой
    записи, и не строит иллюстраций.

    make_illustrations=True — дополнительно сохраняет иллюстрацию (см.
    plot_analysis: фрагмент [0,10]с + RR по всей записи + динамика ЧСС)
    для каждого файла в fig_dir/final_<имя_файла>.png.
    """
    files = sorted(glob.glob(os.path.join(data_dir, "*.edf")))
    if not files:
        print(f"В папке '{data_dir}' не найдено EDF-файлов.")
        return []

    results = []
    if make_illustrations:
        os.makedirs(fig_dir, exist_ok=True)

    for fp in files:
        res_full = analyze_edf(fp, verbose=False)
        r, pk, t = res_full["r_idx"], res_full["p_idx"], res_full["t_idx"]
        hr = 60.0 / np.mean(np.diff(r) / res_full["fs"]) if len(r) > 1 else float("nan")

        row = dict(
            file=os.path.basename(fp), lead=res_full["channel"],
            R=len(r), P=len(pk), T=len(t),
            P_pct=round(100 * len(pk) / len(r)) if len(r) else None,
            T_pct=round(100 * len(t) / len(r)) if len(r) else None,
            mean_hr=round(hr, 1) if len(r) > 1 else None,
            doubled=res_full["double_counting_suspected"],
        )
        results.append(row)
        print(f"{row['file']:15s} канал={row['lead']:6s} R={row['R']:4d} "
              f"P={row['P']:4d}({row['P_pct']:3d}%) T={row['T']:4d}({row['T_pct']:3d}%) "
              f"ЧСС={row['mean_hr']:5.1f} задвоение={'ДА' if row['doubled'] else 'нет'}")

        if make_illustrations:
            res_win = analyze_edf(fp, window_s=window_s, verbose=False)
            save_path = os.path.join(fig_dir, f"final_{os.path.basename(fp).replace('.edf', '')}.png")
            plot_analysis(fp, res_full, res_win, window_s, save_path=save_path)

    if save_json:
        if os.path.dirname(save_json):
            os.makedirs(os.path.dirname(save_json), exist_ok=True)
        with open(save_json, "w") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"\nСохранено: {save_json}")
    return results


# ══════════════════════════════════════════════════════════════════════════
# ТОЧКА ВХОДА
# ══════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════════
# ТОЧКА ВХОДА
# ══════════════════════════════════════════════════════════════════════════

def _strip_jupyter_argv(argv):
    """Убирает служебную пару Jupyter '-f <файл>.json' из argv, если она
    там оказалась (типично при запуске внутри ноутбука через %run)."""
    cleaned = []
    skip_next = False
    for i, a in enumerate(argv):
        if skip_next:
            skip_next = False
            continue
        if a == "-f" and i + 1 < len(argv) and argv[i + 1].endswith(".json"):
            skip_next = True
            continue
        cleaned.append(a)
    return cleaned


def _main():
    import argparse
    import sys

    argv = _strip_jupyter_argv(sys.argv[1:])

    parser = argparse.ArgumentParser(prog="ecg_full_toolkit_FINAL.py")
    sub = parser.add_subparsers(dest="mode")

    p_analyze = sub.add_parser("analyze")
    p_analyze.add_argument("edf_path")
    p_analyze.add_argument("--channel", default=None)

    p_validate = sub.add_parser("validate")
    p_validate.add_argument("data_dir", nargs="?", default=".")

    p_batch = sub.add_parser("batch")
    p_batch.add_argument("data_dir", nargs="?", default=".")
    p_batch.add_argument("--illustrations", action="store_true")

    if argv and argv[0] not in ("analyze", "validate", "batch"):
        if argv[0] == "--validate":
            argv = ["validate"] + argv[1:]
        else:
            argv = ["analyze"] + argv

    args = parser.parse_args(argv)

    if args.mode == "validate":
        run_synthetic_validation()
        run_cross_validation(args.data_dir)

    elif args.mode == "batch":
        run_batch_analysis(args.data_dir, make_illustrations=args.illustrations)

    elif args.mode == "analyze":
        res = analyze_edf(args.edf_path, channel_name=args.channel)
        print(f"R={len(res['r_idx'])} P={len(res['p_idx'])} T={len(res['t_idx'])}")

    else:
        # Без аргументов (в т.ч. в Jupyter после удаления -f kernel.json):
        # если рядом есть EDF-файлы - просто анализируем всё, что нашли.
        edf_files = sorted(glob.glob("*.edf"))
        if edf_files:
            run_batch_analysis(".")
        else:
            print("EDF-файлы не найдены в текущей папке. Использование:")
            print("  python ecg_full_toolkit_FINAL.py файл.edf")
            print("  python ecg_full_toolkit_FINAL.py batch папка/ [--illustrations]")
            print("  python ecg_full_toolkit_FINAL.py validate папка/")


if __name__ == "__main__":
    _main()

0011_ecg.edf    канал=ECG3   R= 716 P= 716(100%) T= 716(100%) ЧСС= 71.0 задвоение=нет
  -> сохранено: figures/final_0011_ecg.png
0012_ecg.edf    канал=ECG3   R= 598 P= 597(100%) T= 598(100%) ЧСС= 59.5 задвоение=нет
  -> сохранено: figures/final_0012_ecg.png
0013_ecg.edf    канал=ECG3   R= 628 P= 627(100%) T= 628(100%) ЧСС= 62.2 задвоение=нет
  -> сохранено: figures/final_0013_ecg.png
0014_ecg.edf    канал=ECG3   R= 710 P= 709(100%) T= 709(100%) ЧСС= 68.8 задвоение=нет
  -> сохранено: figures/final_0014_ecg.png
0015_ecg.edf    канал=ECG3   R= 785 P= 784(100%) T= 784(100%) ЧСС= 76.0 задвоение=нет
  -> сохранено: figures/final_0015_ecg.png
0016_ecg.edf    канал=ECG2   R= 940 P= 939(100%) T= 940(100%) ЧСС= 91.0 задвоение=нет
  -> сохранено: figures/final_0016_ecg.png
0017_ecg.edf    канал=ECG3   R= 813 P= 812(100%) T= 813(100%) ЧСС= 80.2 задвоение=нет
  -> сохранено: figures/final_0017_ecg.png
0018_ecg.edf    канал=ECG2   R= 635 P= 634(100%) T= 635(100%) ЧСС= 63.0 задвоение=нет
  -> сохран